In [9]:
import importlib
import math
import logging
import time
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.cluster import KMeans

from tgn.model.tgn import TGN
from tgn.utils.utils import RandEdgeSampler, get_neighbor_finder
from tgn.utils.data_processing import compute_time_statistics
import tgn.utils.my_data as my_data
import tgn.model.tgn as tgn
import tgn.utils.utils as utils

importlib.reload(my_data)
importlib.reload(tgn)
importlib.reload(utils)

TGN = tgn.TGN
get_neighbor_finder = utils.get_neighbor_finder

torch.manual_seed(0)
np.random.seed(0)

# =========================================================
# 1. 超参数
# =========================================================
BATCH_SIZE = 256
NUM_NEIGHBORS = 10
NUM_NEG = 1
NUM_EPOCH = 100
NUM_HEADS = 2
DROP_OUT = 0.1
data = "p0.8_mu0.2_1"
NUM_LAYER = 3
LEARNING_RATE = 0.001
NODE_DIM = 128
TIME_DIM = 100
USE_MEMORY = False
MESSAGE_DIM = 128
MEMORY_DIM = 128
uniform = False
prefix = ""
backprop_every = 1
message_function = "mlp"
use_source_embedding_in_message = True
use_destination_embedding_in_message = True
aggregator = "mean"
memory_update_at_end = False
embedding_module = "graph_attention"
dyrep = True
memory_updater = "gru"
bidirection = True

N_CLUSTERS = 5
DEC_ALPHA = 1.0
DEC_WARMUP_EPOCHS = 10

# 新增：排序损失相关
K_RANK_NEG = 15
TEACHER_TEMP = 0.1
STUDENT_TEMP = 0.1
LAMBDA_RANK = 1.0
LAMBDA_DEC = 0.1

Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
Path("./results/").mkdir(parents=True, exist_ok=True)
Path("./log/").mkdir(parents=True, exist_ok=True)

MODEL_SAVE_PATH = f"./saved_models/{data}.pth"
get_checkpoint_path = lambda epoch: f"./saved_checkpoints/{data}-{epoch}.pth"
results_path = "results/pretrain_summary_retrieval.pkl" if prefix == "" else f"results/{prefix}_pretrain_summary_retrieval.pkl"

# =========================================================
# 2. logger
# =========================================================
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

fh = logging.FileHandler(f"log/{time.time()}.log")
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
#ch.setLevel(logging.WARN)

formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
fh.setFormatter(formatter)
ch.setFormatter(formatter)

if not logger.handlers:
    logger.addHandler(fh)
    logger.addHandler(ch)
else:
    logger.handlers = []
    logger.addHandler(fh)
    logger.addHandler(ch)

# =========================================================
# 3. 数据：整个 full_data 训练
# =========================================================
node2id_df = pd.read_csv("preprocess/p0.8_mu0.2_1_node2id.csv")
node2id = dict(zip(node2id_df["node"], node2id_df["id"]))

node_features, edge_features, full_data = my_data.get_data(
    data,
    "preprocess/p0.8_mu0.2_1.csv",
    node_embedding_method="one-hot",
    node2id=node2id
)

full_ngh_finder = get_neighbor_finder(full_data, uniform)
rand_sampler = RandEdgeSampler(full_data.sources, full_data.destinations)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
logger.info(f"device = {device}")

all_nodes = np.arange(node_features.shape[0], dtype=np.int64)

# =========================================================
# 4. 工具函数
# =========================================================
def build_pooled_neighbor_summary(
    ngh_finder,
    node_features_np,
    source_nodes,
    timestamps,
    n_neighbors,
    device,
):
    """
    对每个节点在对应时刻构造局部邻居分布摘要。
    在 one-hot node feature 下，这个摘要就是一个加权邻居分布向量。
    """
    source_nodes = np.asarray(source_nodes, dtype=np.int64)
    timestamps = np.asarray(timestamps, dtype=np.float64)

    neighbors, edge_idxs, edge_times = ngh_finder.get_temporal_neighbor_bidirection(
        source_nodes,
        timestamps,
        n_neighbors=n_neighbors
    )

    B = neighbors.shape[0]
    D = node_features_np.shape[1]
    summary = np.zeros((B, D), dtype=np.float32)
    valid_mask = np.zeros(B, dtype=bool)

    for i in range(B):
        nbrs = neighbors[i]
        ts = edge_times[i]

        valid = (ts > 0) & (nbrs >= 0)
        nbrs = nbrs[valid]
        ts = ts[valid]

        if len(nbrs) == 0:
            continue

        valid_mask[i] = True

        dt = np.abs(timestamps[i] - ts)
        beta = 0.01
        weights = np.exp(-beta * dt)
        weights = weights / (weights.sum() + 1e-12)

        summary[i] = (node_features_np[nbrs] * weights[:, None]).sum(axis=0)

    summary_tensor = torch.from_numpy(summary).to(device)
    valid_mask = torch.from_numpy(valid_mask).to(device)
    return summary_tensor, valid_mask


def compute_src_dst_temporal_embeddings(
    tgn,
    sources_batch,
    destinations_batch,
    negatives_batch,
    timestamps_batch,
    edge_idxs_batch,
    n_neighbors,
):
    source_embedding, destination_embedding, negative_embedding = tgn.compute_temporal_embeddings(
        source_nodes=sources_batch,
        destination_nodes=destinations_batch,
        negative_nodes=negatives_batch,
        edge_times=timestamps_batch,
        edge_idxs=edge_idxs_batch,
        n_neighbors=n_neighbors
    )
    return source_embedding, destination_embedding


def sample_candidate_nodes(
    positive_nodes,
    anchor_nodes,
    all_nodes,
    k_neg,
):
    """
    为每个 anchor 构造候选节点集合:
      [positive, neg1, neg2, ..., negk]
    """
    positive_nodes = np.asarray(positive_nodes, dtype=np.int64)
    anchor_nodes = np.asarray(anchor_nodes, dtype=np.int64)
    all_nodes = np.asarray(all_nodes, dtype=np.int64)

    B = len(positive_nodes)
    cand = np.zeros((B, 1 + k_neg), dtype=np.int64)
    cand[:, 0] = positive_nodes

    for i in range(B):
        pos = int(positive_nodes[i])
        anc = int(anchor_nodes[i])

        negs = []
        while len(negs) < k_neg:
            x = int(np.random.choice(all_nodes))
            if x == pos or x == anc:
                continue
            negs.append(x)

        cand[i, 1:] = np.array(negs, dtype=np.int64)

    return cand


def jsd_teacher_scores(anchor_summary, cand_summary, eps=1e-12):
    """
    anchor_summary: [B, D]
    cand_summary:   [B, C, D]
    return:
      teacher_scores: [B, C], 值越大表示越相似，这里定义为 -JSD
    """
    p = anchor_summary / (anchor_summary.sum(dim=1, keepdim=True) + eps)  # [B, D]
    q = cand_summary / (cand_summary.sum(dim=2, keepdim=True) + eps)      # [B, C, D]

    p = p.unsqueeze(1)                                                     # [B, 1, D]
    m = 0.5 * (p + q)                                                      # [B, C, D]

    kl_pm = (p * (torch.log(p + eps) - torch.log(m + eps))).sum(dim=2)    # [B, C]
    kl_qm = (q * (torch.log(q + eps) - torch.log(m + eps))).sum(dim=2)    # [B, C]
    jsd = 0.5 * (kl_pm + kl_qm)                                            # [B, C]

    return -jsd


def listwise_ranking_loss(
    anchor_emb,
    anchor_summary,
    anchor_valid_mask,
    positive_nodes,
    anchor_nodes,
    timestamps,
    ngh_finder,
    node_features_np,
    query_proj,
    summary_proj,
    all_nodes,
    k_neg,
    device,
    teacher_temp=0.2,
    student_temp=0.2,
    n_neighbors=10,
):
    """
    方案2：listwise soft ranking loss

    教师排序：
      由 anchor_summary 和 candidate_summary 的 -JSD 给出

    学生排序：
      由 anchor_emb 和 candidate_summary_repr 的相似度给出
    """
    B, D = anchor_emb.shape
    C = 1 + k_neg

    if B == 0:
        return torch.tensor(0.0, device=device, requires_grad=True)

    # 1) 构造候选节点 [positive + negatives]
    cand_nodes_np = sample_candidate_nodes(
        positive_nodes=positive_nodes,
        anchor_nodes=anchor_nodes,
        all_nodes=all_nodes,
        k_neg=k_neg,
    )  # [B, C]

    # 2) 对候选节点在同一时刻构造局部邻居分布
    cand_nodes_flat = cand_nodes_np.reshape(-1)
    cand_timestamps_flat = np.repeat(np.asarray(timestamps, dtype=np.float64), C)

    cand_summary_flat, cand_valid_flat = build_pooled_neighbor_summary(
        ngh_finder=ngh_finder,
        node_features_np=node_features_np,
        source_nodes=cand_nodes_flat,
        timestamps=cand_timestamps_flat,
        n_neighbors=n_neighbors,
        device=device,
    )

    cand_summary = cand_summary_flat.view(B, C, D)     # [B, C, D]
    cand_valid = cand_valid_flat.view(B, C)            # [B, C]

    # 3) 过滤无效 anchor 行：anchor 有效且 positive 有效
    row_mask = anchor_valid_mask & cand_valid[:, 0]

    if row_mask.sum() == 0:
        return torch.tensor(0.0, device=device, requires_grad=True)

    anchor_emb = anchor_emb[row_mask]                  # [M, D]
    anchor_summary = anchor_summary[row_mask]          # [M, D]
    cand_summary = cand_summary[row_mask]              # [M, C, D]
    cand_valid = cand_valid[row_mask]                  # [M, C]

    # 4) 教师分数：-JSD
    teacher_scores = jsd_teacher_scores(anchor_summary, cand_summary)  # [M, C]
    teacher_scores = teacher_scores.masked_fill(~cand_valid, -1e9)
    teacher_q = F.softmax(teacher_scores / teacher_temp, dim=1)

    # 5) 学生分数：anchor_emb vs candidate_summary_repr
    q_anchor = query_proj(anchor_emb)                                  # [M, D]
    k_cand = summary_proj(cand_summary.view(-1, D)).view(cand_summary.shape[0], C, D)  # [M, C, D]

    q_anchor = F.normalize(q_anchor, dim=1)
    k_cand = F.normalize(k_cand, dim=2)

    student_scores = (q_anchor.unsqueeze(1) * k_cand).sum(dim=2)      # [M, C]
    student_scores = student_scores.masked_fill(~cand_valid, -1e9)
    student_log_q = F.log_softmax(student_scores / student_temp, dim=1)

    # 6) KL(teacher || student)
    loss = F.kl_div(student_log_q, teacher_q, reduction="batchmean")
    return loss


class DECLayer(torch.nn.Module):
    def __init__(self, n_clusters, hidden_dim, alpha=1.0):
        super().__init__()
        self.n_clusters = n_clusters
        self.hidden_dim = hidden_dim
        self.alpha = alpha
        self.cluster_centers = torch.nn.Parameter(torch.randn(n_clusters, hidden_dim))

    def soft_assign(self, z):
        dist_sq = torch.sum((z.unsqueeze(1) - self.cluster_centers.unsqueeze(0)) ** 2, dim=2)
        q = 1.0 / (1.0 + dist_sq / self.alpha)
        q = q ** ((self.alpha + 1.0) / 2.0)
        q = q / torch.sum(q, dim=1, keepdim=True)
        return q

    @staticmethod
    def target_distribution(q):
        weight = q ** 2 / torch.sum(q, dim=0, keepdim=True)
        p = weight / torch.sum(weight, dim=1, keepdim=True)
        return p

    def dec_loss(self, z):
        q = self.soft_assign(z)
        p = self.target_distribution(q).detach()
        loss = F.kl_div(torch.log(q + 1e-12), p, reduction="batchmean")
        return loss, q, p


def eval_summary_retrieval(
    model,
    query_proj,
    summary_proj,
    ngh_finder,
    data,
    rand_sampler,
    node_features_np,
    batch_size,
    n_neighbors,
    device,
    temperature=0.2,
    topk=5,
):
    model.eval()
    query_proj.eval()
    summary_proj.eval()

    num_instance = len(data.sources)
    num_batch = math.ceil(num_instance / batch_size)

    total_valid = 0
    total_top1 = 0.0
    total_topk = 0.0
    total_mrr = 0.0

    with torch.no_grad():
        for k in range(num_batch):
            start_idx = k * batch_size
            end_idx = min(num_instance, start_idx + batch_size)

            sources_batch = data.sources[start_idx:end_idx]
            destinations_batch = data.destinations[start_idx:end_idx]
            edge_idxs_batch = data.edge_idxs[start_idx:end_idx]
            timestamps_batch = data.timestamps[start_idx:end_idx]

            size = len(sources_batch)
            if size == 0:
                continue

            _, negatives_batch = rand_sampler.sample(size)

            src_emb, dst_emb = compute_src_dst_temporal_embeddings(
                tgn=model,
                sources_batch=sources_batch,
                destinations_batch=destinations_batch,
                negatives_batch=negatives_batch,
                timestamps_batch=timestamps_batch,
                edge_idxs_batch=edge_idxs_batch,
                n_neighbors=n_neighbors
            )

            src_summary, src_valid_mask = build_pooled_neighbor_summary(
                ngh_finder=ngh_finder,
                node_features_np=node_features_np,
                source_nodes=sources_batch,
                timestamps=timestamps_batch,
                n_neighbors=n_neighbors,
                device=device,
            )

            dst_summary, dst_valid_mask = build_pooled_neighbor_summary(
                ngh_finder=ngh_finder,
                node_features_np=node_features_np,
                source_nodes=destinations_batch,
                timestamps=timestamps_batch,
                n_neighbors=n_neighbors,
                device=device,
            )

            emb_list = []
            sum_list = []

            if src_valid_mask.sum().item() > 0:
                emb_list.append(src_emb[src_valid_mask])
                sum_list.append(src_summary[src_valid_mask])

            if dst_valid_mask.sum().item() > 0:
                emb_list.append(dst_emb[dst_valid_mask])
                sum_list.append(dst_summary[dst_valid_mask])

            if len(emb_list) == 0:
                continue

            z_u = torch.cat(emb_list, dim=0)
            pooled_summary = torch.cat(sum_list, dim=0)

            q = query_proj(z_u)
            s = summary_proj(pooled_summary)

            q = F.normalize(q, dim=1)
            s = F.normalize(s, dim=1)

            logits = q @ s.t() / temperature
            M = logits.size(0)
            labels = torch.arange(M, device=logits.device)

            pred_top1 = torch.argmax(logits, dim=1)
            top1_correct = (pred_top1 == labels).float().sum().item()

            k_eff = min(topk, M)
            topk_idx = torch.topk(logits, k=k_eff, dim=1).indices
            topk_correct = (topk_idx == labels.unsqueeze(1)).any(dim=1).float().sum().item()

            sorted_idx = torch.argsort(logits, dim=1, descending=True)
            match_pos = (sorted_idx == labels.unsqueeze(1)).nonzero(as_tuple=False)
            ranks = match_pos[:, 1].float() + 1.0
            mrr = (1.0 / ranks).sum().item()

            total_valid += M
            total_top1 += top1_correct
            total_topk += topk_correct
            total_mrr += mrr

    if total_valid == 0:
        return {"top1": 0.0, "topk": 0.0, "mrr": 0.0, "num_valid": 0}

    return {
        "top1": total_top1 / total_valid,
        "topk": total_topk / total_valid,
        "mrr": total_mrr / total_valid,
        "num_valid": total_valid,
    }


def collect_embeddings_for_dec_init(
    model,
    data,
    rand_sampler,
    batch_size,
    n_neighbors,
    max_samples=5000,
):
    model.eval()
    all_emb = []

    num_instance = len(data.sources)
    num_batch = math.ceil(num_instance / batch_size)

    with torch.no_grad():
        for k in range(num_batch):
            start_idx = k * batch_size
            end_idx = min(num_instance, start_idx + batch_size)

            sources_batch = data.sources[start_idx:end_idx]
            destinations_batch = data.destinations[start_idx:end_idx]
            edge_idxs_batch = data.edge_idxs[start_idx:end_idx]
            timestamps_batch = data.timestamps[start_idx:end_idx]

            size = len(sources_batch)
            if size == 0:
                continue

            _, negatives_batch = rand_sampler.sample(size)

            src_emb, dst_emb = compute_src_dst_temporal_embeddings(
                tgn=model,
                sources_batch=sources_batch,
                destinations_batch=destinations_batch,
                negatives_batch=negatives_batch,
                timestamps_batch=timestamps_batch,
                edge_idxs_batch=edge_idxs_batch,
                n_neighbors=n_neighbors
            )

            emb = torch.cat([src_emb, dst_emb], dim=0)
            all_emb.append(emb.detach().cpu())

            if sum(x.shape[0] for x in all_emb) >= max_samples:
                break

    if len(all_emb) == 0:
        return None

    X = torch.cat(all_emb, dim=0).numpy()
    if X.shape[0] > max_samples:
        X = X[:max_samples]
    return X


def init_dec_centers_with_kmeans(dec_layer, X):
    km = KMeans(n_clusters=dec_layer.n_clusters, n_init=20, random_state=0)
    km.fit(X)
    centers = torch.tensor(km.cluster_centers_, dtype=torch.float32, device=dec_layer.cluster_centers.device)
    with torch.no_grad():
        dec_layer.cluster_centers.copy_(centers)


# =========================================================
# 5. time statistics
# =========================================================
mean_time_shift_src, std_time_shift_src, mean_time_shift_dst, std_time_shift_dst = \
    compute_time_statistics(full_data.sources, full_data.destinations, full_data.timestamps)

# =========================================================
# 6. model
# =========================================================
emb_dim = node_features.shape[1]

dec_layer = DECLayer(
    n_clusters=N_CLUSTERS,
    hidden_dim=emb_dim,
    alpha=DEC_ALPHA
).to(device)

tgn = TGN(
    neighbor_finder=full_ngh_finder,
    node_features=node_features,
    edge_features=edge_features,
    device=device,
    n_layers=NUM_LAYER,
    n_heads=NUM_HEADS,
    dropout=DROP_OUT,
    use_memory=USE_MEMORY,
    message_dimension=MESSAGE_DIM,
    memory_dimension=MEMORY_DIM,
    memory_update_at_start=not memory_update_at_end,
    embedding_module_type=embedding_module,
    message_function=message_function,
    aggregator_type=aggregator,
    memory_updater_type=memory_updater,
    n_neighbors=NUM_NEIGHBORS,
    mean_time_shift_src=mean_time_shift_src,
    std_time_shift_src=std_time_shift_src,
    mean_time_shift_dst=mean_time_shift_dst,
    std_time_shift_dst=std_time_shift_dst,
    use_destination_embedding_in_message=use_destination_embedding_in_message,
    use_source_embedding_in_message=use_source_embedding_in_message,
    dyrep=dyrep,
    bidirection=bidirection
).to(device)

proj_dim = node_features.shape[1]
query_proj = torch.nn.Sequential(
    torch.nn.Linear(node_features.shape[1], proj_dim),
    torch.nn.ReLU(),
    torch.nn.Linear(proj_dim, proj_dim),
).to(device)

summary_proj = torch.nn.Sequential(
    torch.nn.Linear(node_features.shape[1], proj_dim),
    torch.nn.ReLU(),
    torch.nn.Linear(proj_dim, proj_dim),
).to(device)

temperature = 0.2

optimizer = torch.optim.AdamW(
    list(tgn.parameters()) +
    list(query_proj.parameters()) +
    list(summary_proj.parameters()) +
    list(dec_layer.parameters()),
    lr=LEARNING_RATE
)

num_instance = len(full_data.sources)
num_batch = math.ceil(num_instance / BATCH_SIZE)

logger.info(f"num of training instances: {num_instance}")
logger.info(f"num of batches per epoch: {num_batch}")

retrieval_top1s = []
retrieval_topks = []
retrieval_mrrs = []
epoch_times = []
total_epoch_times = []
train_losses = []
train_rank_losses = []
train_dec_losses = []

# =========================================================
# 7. init DEC
# =========================================================
logger.info("Collecting embeddings for DEC initialization...")
X_init = collect_embeddings_for_dec_init(
    model=tgn,
    data=full_data,
    rand_sampler=rand_sampler,
    batch_size=BATCH_SIZE,
    n_neighbors=NUM_NEIGHBORS,
    max_samples=5000,
)

if X_init is not None and X_init.shape[0] >= N_CLUSTERS:
    init_dec_centers_with_kmeans(dec_layer, X_init)
    logger.info("DEC centers initialized with KMeans.")
else:
    logger.warning("DEC initialization skipped: not enough samples.")

# =========================================================
# 8. 训练：整个 full_data
# =========================================================
for epoch in range(NUM_EPOCH):
    start_epoch = time.time()

    if USE_MEMORY:
        tgn.memory.__init_memory__()

    tgn.set_neighbor_finder(full_ngh_finder)

    m_loss = []
    m_rank_loss = []
    m_dec_loss = []

    logger.info(f"start epoch {epoch}")

    for k in range(0, num_batch, backprop_every):
        loss = 0.0
        optimizer.zero_grad()

        batch_rank_vals = []
        batch_dec_vals = []

        for j in range(backprop_every):
            batch_idx = k + j
            if batch_idx >= num_batch:
                continue

            start_idx = batch_idx * BATCH_SIZE
            end_idx = min(num_instance, start_idx + BATCH_SIZE)

            sources_batch = full_data.sources[start_idx:end_idx]
            destinations_batch = full_data.destinations[start_idx:end_idx]
            edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]
            timestamps_batch = full_data.timestamps[start_idx:end_idx]

            size = len(sources_batch)
            _, negatives_batch = rand_sampler.sample(size)

            tgn.train()
            query_proj.train()
            summary_proj.train()
            dec_layer.train()

            src_emb, dst_emb = compute_src_dst_temporal_embeddings(
                tgn=tgn,
                sources_batch=sources_batch,
                destinations_batch=destinations_batch,
                negatives_batch=negatives_batch,
                timestamps_batch=timestamps_batch,
                edge_idxs_batch=edge_idxs_batch,
                n_neighbors=NUM_NEIGHBORS
            )

            src_summary, src_valid_mask = build_pooled_neighbor_summary(
                ngh_finder=full_ngh_finder,
                node_features_np=node_features,
                source_nodes=sources_batch,
                timestamps=timestamps_batch,
                n_neighbors=NUM_NEIGHBORS,
                device=device,
            )

            dst_summary, dst_valid_mask = build_pooled_neighbor_summary(
                ngh_finder=full_ngh_finder,
                node_features_np=node_features,
                source_nodes=destinations_batch,
                timestamps=timestamps_batch,
                n_neighbors=NUM_NEIGHBORS,
                device=device,
            )

            src_rank_loss = listwise_ranking_loss(
                anchor_emb=src_emb,
                anchor_summary=src_summary,
                anchor_valid_mask=src_valid_mask,
                positive_nodes=destinations_batch,
                anchor_nodes=sources_batch,
                timestamps=timestamps_batch,
                ngh_finder=full_ngh_finder,
                node_features_np=node_features,
                query_proj=query_proj,
                summary_proj=summary_proj,
                all_nodes=all_nodes,
                k_neg=K_RANK_NEG,
                device=device,
                teacher_temp=TEACHER_TEMP,
                student_temp=STUDENT_TEMP,
                n_neighbors=NUM_NEIGHBORS,
            )

            dst_rank_loss = listwise_ranking_loss(
                anchor_emb=dst_emb,
                anchor_summary=dst_summary,
                anchor_valid_mask=dst_valid_mask,
                positive_nodes=sources_batch,
                anchor_nodes=destinations_batch,
                timestamps=timestamps_batch,
                ngh_finder=full_ngh_finder,
                node_features_np=node_features,
                query_proj=query_proj,
                summary_proj=summary_proj,
                all_nodes=all_nodes,
                k_neg=K_RANK_NEG,
                device=device,
                teacher_temp=TEACHER_TEMP,
                student_temp=STUDENT_TEMP,
                n_neighbors=NUM_NEIGHBORS,
            )

            rank_loss = 0.5 * (src_rank_loss + dst_rank_loss)

            batch_emb = torch.cat([src_emb, dst_emb], dim=0)

            if epoch >= DEC_WARMUP_EPOCHS:
                dec_loss, q_batch, p_batch = dec_layer.dec_loss(batch_emb)
            else:
                dec_loss = torch.tensor(0.0, device=device)

            total_batch_loss = LAMBDA_RANK * rank_loss + LAMBDA_DEC * dec_loss
            loss += total_batch_loss

            batch_rank_vals.append(rank_loss.item())
            batch_dec_vals.append(dec_loss.item())

        loss = loss / backprop_every
        loss.backward()
        optimizer.step()

        m_loss.append(loss.item())
        m_rank_loss.append(float(np.mean(batch_rank_vals)) if len(batch_rank_vals) > 0 else 0.0)
        m_dec_loss.append(float(np.mean(batch_dec_vals)) if len(batch_dec_vals) > 0 else 0.0)

        if USE_MEMORY:
            tgn.memory.detach_memory()

    epoch_time = time.time() - start_epoch
    epoch_times.append(epoch_time)

    retrieval_metrics = eval_summary_retrieval(
        model=tgn,
        query_proj=query_proj,
        summary_proj=summary_proj,
        ngh_finder=full_ngh_finder,
        data=full_data,
        rand_sampler=rand_sampler,
        node_features_np=node_features,
        batch_size=BATCH_SIZE,
        n_neighbors=NUM_NEIGHBORS,
        device=device,
        temperature=temperature,
        topk=5,
    )

    retrieval_top1s.append(retrieval_metrics["top1"])
    retrieval_topks.append(retrieval_metrics["topk"])
    retrieval_mrrs.append(retrieval_metrics["mrr"])

    train_losses.append(np.mean(m_loss))
    train_rank_losses.append(np.mean(m_rank_loss))
    train_dec_losses.append(np.mean(m_dec_loss))

    total_epoch_time = time.time() - start_epoch
    total_epoch_times.append(total_epoch_time)

    pickle.dump({
        "retrieval_top1s": retrieval_top1s,
        "retrieval_topks": retrieval_topks,
        "retrieval_mrrs": retrieval_mrrs,
        "train_losses": train_losses,
        "train_rank_losses": train_rank_losses,
        "train_dec_losses": train_dec_losses,
        "epoch_times": epoch_times,
        "total_epoch_times": total_epoch_times
    }, open(results_path, "wb"))

    logger.info(f"epoch: {epoch} took {total_epoch_time:.2f}s")
    logger.info(f"Epoch mean total loss: {np.mean(m_loss)}")
    logger.info(f"Epoch mean rank loss: {np.mean(m_rank_loss)}")
    logger.info(f"Epoch mean dec loss: {np.mean(m_dec_loss)}")
    logger.info(
        f"retrieval top1: {retrieval_metrics['top1']:.4f}, "
        f"top5: {retrieval_metrics['topk']:.4f}, "
        f"mrr: {retrieval_metrics['mrr']:.4f}"
    )

    torch.save({
        "tgn": tgn.state_dict(),
        "query_proj": query_proj.state_dict(),
        "summary_proj": summary_proj.state_dict(),
        "dec_layer": dec_layer.state_dict(),
        "config": {
            "NUM_NEIGHBORS": NUM_NEIGHBORS,
            "NUM_LAYER": NUM_LAYER,
            "NUM_HEADS": NUM_HEADS,
            "DROP_OUT": DROP_OUT,
            "USE_MEMORY": USE_MEMORY,
            "MESSAGE_DIM": MESSAGE_DIM,
            "MEMORY_DIM": MEMORY_DIM,
            "embedding_module": embedding_module,
            "message_function": message_function,
            "aggregator": aggregator,
            "memory_updater": memory_updater,
            "use_destination_embedding_in_message": use_destination_embedding_in_message,
            "use_source_embedding_in_message": use_source_embedding_in_message,
            "dyrep": dyrep,
            "bidirection": bidirection,
            "N_CLUSTERS": N_CLUSTERS,
            "DEC_ALPHA": DEC_ALPHA,
            "K_RANK_NEG": K_RANK_NEG,
            "TEACHER_TEMP": TEACHER_TEMP,
            "STUDENT_TEMP": STUDENT_TEMP,
            "LAMBDA_RANK": LAMBDA_RANK,
            "LAMBDA_DEC": LAMBDA_DEC,
        }
    }, get_checkpoint_path(epoch))

# =========================================================
# 9. 最终保存
# =========================================================
logger.info("Saving final model...")

torch.save({
    "tgn": tgn.state_dict(),
    "query_proj": query_proj.state_dict(),
    "summary_proj": summary_proj.state_dict(),
    "dec_layer": dec_layer.state_dict(),
    "config": {
        "NUM_NEIGHBORS": NUM_NEIGHBORS,
        "NUM_LAYER": NUM_LAYER,
        "NUM_HEADS": NUM_HEADS,
        "DROP_OUT": DROP_OUT,
        "USE_MEMORY": USE_MEMORY,
        "MESSAGE_DIM": MESSAGE_DIM,
        "MEMORY_DIM": MEMORY_DIM,
        "embedding_module": embedding_module,
        "message_function": message_function,
        "aggregator": aggregator,
        "memory_updater": memory_updater,
        "use_destination_embedding_in_message": use_destination_embedding_in_message,
        "use_source_embedding_in_message": use_source_embedding_in_message,
        "dyrep": dyrep,
        "bidirection": bidirection,
        "N_CLUSTERS": N_CLUSTERS,
        "DEC_ALPHA": DEC_ALPHA,
        "K_RANK_NEG": K_RANK_NEG,
        "TEACHER_TEMP": TEACHER_TEMP,
        "STUDENT_TEMP": STUDENT_TEMP,
        "LAMBDA_RANK": LAMBDA_RANK,
        "LAMBDA_DEC": LAMBDA_DEC,
    }
}, MODEL_SAVE_PATH)

logger.info(f"TGN model saved to {MODEL_SAVE_PATH}")

2026-03-22 22:43:31,614 - root - INFO - device = mps


cannot find (./data/p0.8_mu0.2_1.npy), use zero-vector for edge feat (dim=16)...
Use one-hot init for node embedding. 
The dataset has 457 interactions, involving 50 different nodes


2026-03-22 22:43:31,919 - root - INFO - num of training instances: 457
2026-03-22 22:43:31,919 - root - INFO - num of batches per epoch: 2
2026-03-22 22:43:31,920 - root - INFO - Collecting embeddings for DEC initialization...
2026-03-22 22:43:46,828 - root - INFO - DEC centers initialized with KMeans.
2026-03-22 22:43:46,840 - root - INFO - start epoch 0
2026-03-22 22:43:51,140 - root - INFO - epoch: 0 took 4.27s
2026-03-22 22:43:51,150 - root - INFO - Epoch mean total loss: 0.7350898683071136
2026-03-22 22:43:51,151 - root - INFO - Epoch mean rank loss: 0.7350898683071136
2026-03-22 22:43:51,152 - root - INFO - Epoch mean dec loss: 0.0
2026-03-22 22:43:51,152 - root - INFO - retrieval top1: 0.0022, top5: 0.0120, mrr: 0.0158
2026-03-22 22:43:51,200 - root - INFO - start epoch 1
2026-03-22 22:43:54,268 - root - INFO - epoch: 1 took 2.95s
2026-03-22 22:43:54,271 - root - INFO - Epoch mean total loss: 0.7388584911823273
2026-03-22 22:43:54,271 - root - INFO - Epoch mean rank loss: 0.7388

In [10]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import tgn.utils.my_data as my_data
from tgn.utils.data_processing import compute_time_statistics
from tgn.utils.utils import get_neighbor_finder
from tgn.model.tgn import TGN


# =========================================================
# 0. DEC layer
# =========================================================
class DECLayer(torch.nn.Module):
    def __init__(self, n_clusters, hidden_dim, alpha=1.0):
        super().__init__()
        self.n_clusters = n_clusters
        self.hidden_dim = hidden_dim
        self.alpha = alpha
        self.cluster_centers = torch.nn.Parameter(
            torch.randn(n_clusters, hidden_dim)
        )

    def soft_assign(self, z):
        """
        z: [B, D]
        return q: [B, K]
        """
        dist_sq = torch.sum(
            (z.unsqueeze(1) - self.cluster_centers.unsqueeze(0)) ** 2,
            dim=2
        )  # [B, K]

        q = 1.0 / (1.0 + dist_sq / self.alpha)
        q = q ** ((self.alpha + 1.0) / 2.0)
        q = q / torch.sum(q, dim=1, keepdim=True)
        return q


# =========================================================
# 1. 基本配置
# =========================================================
FILE_NAME = "p0.8_mu0.2_1"
MODEL_PATH = f"saved_models/{FILE_NAME}.pth"
NODE2ID_PATH = f"preprocess/{FILE_NAME}_node2id.csv"
OUT_DIR = Path(f"results/{FILE_NAME}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)


# =========================================================
# 2. 读取 checkpoint
# =========================================================
ckpt = torch.load(MODEL_PATH, map_location=device)

if not isinstance(ckpt, dict):
    raise ValueError(
        f"{MODEL_PATH} 不是完整 checkpoint 字典。"
        f"当前导出脚本要求 checkpoint 至少包含 tgn / dec_layer / config。"
    )

required_keys = ["tgn", "dec_layer", "config"]
missing_keys = [k for k in required_keys if k not in ckpt]
if len(missing_keys) > 0:
    raise ValueError(f"Checkpoint 缺少必要字段: {missing_keys}")

config = ckpt["config"]
print("Loaded checkpoint config keys:", list(config.keys()))


# =========================================================
# 3. 从 checkpoint 恢复训练参数
# =========================================================
BATCH_SIZE = config.get("BATCH_SIZE", 256)
NUM_NEIGHBORS = config["NUM_NEIGHBORS"]
NUM_LAYER = config["NUM_LAYER"]
NUM_HEADS = config["NUM_HEADS"]
DROP_OUT = config["DROP_OUT"]
USE_MEMORY = config["USE_MEMORY"]
MESSAGE_DIM = config["MESSAGE_DIM"]
MEMORY_DIM = config["MEMORY_DIM"]
embedding_module = config["embedding_module"]
message_function = config["message_function"]
aggregator = config["aggregator"]
memory_updater = config["memory_updater"]
use_destination_embedding_in_message = config["use_destination_embedding_in_message"]
use_source_embedding_in_message = config["use_source_embedding_in_message"]
dyrep = config["dyrep"]
bidirection = config["bidirection"]

N_CLUSTERS = config["N_CLUSTERS"]
DEC_ALPHA = config["DEC_ALPHA"]

memory_update_at_end = config.get("memory_update_at_end", False)
memory_update_at_start = not memory_update_at_end

# 新训练脚本里建议把这两个也存进 config
data_csv_path = config.get("data_csv_path", f"preprocess/{FILE_NAME}.csv")
node_embedding_method = config.get("node_embedding_method", "one-hot")

print("Recovered config:")
print("NUM_NEIGHBORS =", NUM_NEIGHBORS)
print("USE_MEMORY =", USE_MEMORY)
print("N_CLUSTERS =", N_CLUSTERS)
print("DEC_ALPHA =", DEC_ALPHA)
print("data_csv_path =", data_csv_path)
print("node_embedding_method =", node_embedding_method)


# =========================================================
# 4. 读取 node2id，并构造映射
# =========================================================
node2id_df = pd.read_csv(NODE2ID_PATH)

if {"node", "id"}.issubset(node2id_df.columns):
    node2id = dict(zip(node2id_df["node"], node2id_df["id"]))
    id_to_node = dict(zip(node2id_df["id"], node2id_df["node"]))
elif node2id_df.shape[1] >= 2:
    col0, col1 = node2id_df.columns[:2]

    if "id" in col0.lower():
        id_col, node_col = col0, col1
    elif "id" in col1.lower():
        id_col, node_col = col1, col0
    else:
        node_col, id_col = col0, col1

    node2id = dict(zip(node2id_df[node_col], node2id_df[id_col]))
    id_to_node = dict(zip(node2id_df[id_col], node2id_df[node_col]))
else:
    raise ValueError(f"Unexpected node2id file format: {NODE2ID_PATH}")

print(f"Loaded node2id from {NODE2ID_PATH}, total {len(id_to_node)} nodes")


# =========================================================
# 5. 读取数据（必须和训练时一致）
# =========================================================
node_features, edge_features, full_data = my_data.get_data(
    FILE_NAME,
    data_csv_path,
    node_embedding_method=node_embedding_method,
    node2id=node2id
)

full_ngh_finder = get_neighbor_finder(full_data, uniform=False)

mean_time_shift_src, std_time_shift_src, mean_time_shift_dst, std_time_shift_dst = \
    compute_time_statistics(
        full_data.sources,
        full_data.destinations,
        full_data.timestamps
    )

print("mean_time_shift_src =", mean_time_shift_src)
print("std_time_shift_src  =", std_time_shift_src)
print("mean_time_shift_dst =", mean_time_shift_dst)
print("std_time_shift_dst  =", std_time_shift_dst)


# =========================================================
# 6. 重建 TGN
# =========================================================
tgn = TGN(
    neighbor_finder=full_ngh_finder,
    node_features=node_features,
    edge_features=edge_features,
    device=device,
    n_layers=NUM_LAYER,
    n_heads=NUM_HEADS,
    dropout=DROP_OUT,
    use_memory=USE_MEMORY,
    message_dimension=MESSAGE_DIM,
    memory_dimension=MEMORY_DIM,
    memory_update_at_start=memory_update_at_start,
    embedding_module_type=embedding_module,
    message_function=message_function,
    aggregator_type=aggregator,
    memory_updater_type=memory_updater,
    n_neighbors=NUM_NEIGHBORS,
    mean_time_shift_src=mean_time_shift_src,
    std_time_shift_src=std_time_shift_src,
    mean_time_shift_dst=mean_time_shift_dst,
    std_time_shift_dst=std_time_shift_dst,
    use_destination_embedding_in_message=use_destination_embedding_in_message,
    use_source_embedding_in_message=use_source_embedding_in_message,
    dyrep=dyrep,
    bidirection=bidirection,
).to(device)

tgn.load_state_dict(ckpt["tgn"], strict=True)
tgn.eval()
print("TGN loaded successfully.")

if USE_MEMORY and hasattr(tgn, "memory") and tgn.memory is not None:
    if hasattr(tgn.memory, "__init_memory__"):
        tgn.memory.__init_memory__()
    elif hasattr(tgn.memory, "reset_memory"):
        tgn.memory.reset_memory()


# =========================================================
# 7. 重建 projection heads（虽然导出 DEC 不一定用到，但和 checkpoint 对齐）
# =========================================================
proj_dim = node_features.shape[1]

query_proj = torch.nn.Sequential(
    torch.nn.Linear(node_features.shape[1], proj_dim),
    torch.nn.ReLU(),
    torch.nn.Linear(proj_dim, proj_dim),
).to(device)

summary_proj = torch.nn.Sequential(
    torch.nn.Linear(node_features.shape[1], proj_dim),
    torch.nn.ReLU(),
    torch.nn.Linear(proj_dim, proj_dim),
).to(device)

if "query_proj" in ckpt:
    query_proj.load_state_dict(ckpt["query_proj"], strict=True)
    query_proj.eval()
    print("query_proj loaded successfully.")
else:
    print("Warning: query_proj not found in checkpoint, skipped.")

if "summary_proj" in ckpt:
    summary_proj.load_state_dict(ckpt["summary_proj"], strict=True)
    summary_proj.eval()
    print("summary_proj loaded successfully.")
else:
    print("Warning: summary_proj not found in checkpoint, skipped.")


# =========================================================
# 8. 重建 DEC layer
# =========================================================
emb_dim = node_features.shape[1]

dec_layer = DECLayer(
    n_clusters=N_CLUSTERS,
    hidden_dim=emb_dim,
    alpha=DEC_ALPHA
).to(device)

dec_layer.load_state_dict(ckpt["dec_layer"], strict=True)
dec_layer.eval()
print("DEC layer loaded successfully.")


# =========================================================
# 9. 导出 node-event embedding + DEC soft assignment
# =========================================================
all_src_emb = []
all_dst_emb = []
all_src_q = []
all_dst_q = []
all_meta = []

num_instance = len(full_data.sources)
num_batch = int(np.ceil(num_instance / BATCH_SIZE))

with torch.no_grad():
    for batch_idx in range(num_batch):
        start = batch_idx * BATCH_SIZE
        end = min(num_instance, start + BATCH_SIZE)

        sources_batch = full_data.sources[start:end]
        destinations_batch = full_data.destinations[start:end]
        timestamps_batch = full_data.timestamps[start:end]
        edge_idxs_batch = full_data.edge_idxs[start:end]

        # 这里只是接口占位，和训练时保持一致
        negative_nodes_batch = destinations_batch

        src_emb_t, dst_emb_t, _ = tgn.compute_temporal_embeddings(
            source_nodes=sources_batch,
            destination_nodes=destinations_batch,
            negative_nodes=negative_nodes_batch,
            edge_times=timestamps_batch,
            edge_idxs=edge_idxs_batch,
            n_neighbors=NUM_NEIGHBORS
        )

        src_q_t = dec_layer.soft_assign(src_emb_t)
        dst_q_t = dec_layer.soft_assign(dst_emb_t)

        src_emb_np = src_emb_t.detach().cpu().numpy().astype(np.float32)
        dst_emb_np = dst_emb_t.detach().cpu().numpy().astype(np.float32)
        src_q_np = src_q_t.detach().cpu().numpy().astype(np.float32)
        dst_q_np = dst_q_t.detach().cpu().numpy().astype(np.float32)

        all_src_emb.append((src_emb_np))
        all_dst_emb.append(dst_emb_np)
        all_src_q.append(src_q_np)
        all_dst_q.append(dst_q_np)

        batch_meta = pd.DataFrame({
            "source_id": sources_batch,
            "destination_id": destinations_batch,
            "timestamp": timestamps_batch,
            "edge_idx": edge_idxs_batch
        })
        all_meta.append(batch_meta)

        if batch_idx % 20 == 0 or batch_idx == num_batch - 1:
            print(f"[{batch_idx + 1}/{num_batch}] done")

src_emb = np.concatenate(all_src_emb, axis=0)
dst_emb = np.concatenate(all_dst_emb, axis=0)
src_q = np.concatenate(all_src_q, axis=0)
dst_q = np.concatenate(all_dst_q, axis=0)
meta = pd.concat(all_meta, axis=0, ignore_index=True)

print("src_emb shape:", src_emb.shape)
print("dst_emb shape:", dst_emb.shape)
print("src_q shape  :", src_q.shape)
print("dst_q shape  :", dst_q.shape)
print("meta shape   :", meta.shape)


# =========================================================
# 10. 映射回原始 node
# =========================================================
meta["source"] = meta["source_id"].map(id_to_node)
meta["destination"] = meta["destination_id"].map(id_to_node)

if meta["source"].isna().any() or meta["destination"].isna().any():
    bad_src = meta.loc[meta["source"].isna(), "source_id"].unique().tolist()
    bad_dst = meta.loc[meta["destination"].isna(), "destination_id"].unique().tolist()
    raise ValueError(
        f"Some node ids cannot be mapped back.\n"
        f"Bad source ids: {bad_src[:10]}\n"
        f"Bad destination ids: {bad_dst[:10]}"
    )


# =========================================================
# 11. 保存 embedding / soft assignment / meta
# =========================================================
np.save(OUT_DIR / "full_src.npy", src_emb)
np.save(OUT_DIR / "full_dst.npy", dst_emb)
np.save(OUT_DIR / "full_src_q.npy", src_q)
np.save(OUT_DIR / "full_dst_q.npy", dst_q)

meta[["source", "destination", "timestamp", "edge_idx"]].to_csv(
    OUT_DIR / "node_event_meta.csv",
    index=False
)

print("Saved:")
print(OUT_DIR / "full_src.npy")
print(OUT_DIR / "full_dst.npy")
print(OUT_DIR / "full_src_q.npy")
print(OUT_DIR / "full_dst_q.npy")
print(OUT_DIR / "node_event_meta.csv")


# =========================================================
# 12. 用 DEC 直接输出聚类标签
# =========================================================
src_labels = np.argmax(src_q, axis=1)
dst_labels = np.argmax(dst_q, axis=1)

result_df = meta[["source", "destination", "timestamp"]].copy()
result_df["source_commu"] = src_labels
result_df["destination_commu"] = dst_labels

result_df.to_csv(OUT_DIR / "tgn_dec.csv", index=False)

print("Saved DEC clustering result to:")
print(OUT_DIR / "tgn_dec.csv")
print(result_df.head())


# =========================================================
# 13. 保存带 confidence 的结果
# =========================================================
result_prob_df = result_df.copy()
result_prob_df["source_confidence"] = src_q.max(axis=1)
result_prob_df["destination_confidence"] = dst_q.max(axis=1)

result_prob_df.to_csv(OUT_DIR / "tgn_dec_with_confidence.csv", index=False)

print("Saved DEC clustering result with confidence to:")
print(OUT_DIR / "tgn_dec.csv")
print(result_prob_df.head())

Using device: mps
Loaded checkpoint config keys: ['NUM_NEIGHBORS', 'NUM_LAYER', 'NUM_HEADS', 'DROP_OUT', 'USE_MEMORY', 'MESSAGE_DIM', 'MEMORY_DIM', 'embedding_module', 'message_function', 'aggregator', 'memory_updater', 'use_destination_embedding_in_message', 'use_source_embedding_in_message', 'dyrep', 'bidirection', 'N_CLUSTERS', 'DEC_ALPHA', 'K_RANK_NEG', 'TEACHER_TEMP', 'STUDENT_TEMP', 'LAMBDA_RANK', 'LAMBDA_DEC']
Recovered config:
NUM_NEIGHBORS = 10
USE_MEMORY = False
N_CLUSTERS = 5
DEC_ALPHA = 1.0
data_csv_path = preprocess/p0.8_mu0.2_1.csv
node_embedding_method = one-hot
Loaded node2id from preprocess/p0.8_mu0.2_1_node2id.csv, total 50 nodes
cannot find (./data/p0.8_mu0.2_1.npy), use zero-vector for edge feat (dim=16)...
Use one-hot init for node embedding. 
The dataset has 457 interactions, involving 50 different nodes
mean_time_shift_src = 246.17943107221006
std_time_shift_src  = 301.34130233998195
mean_time_shift_dst = 237.96498905908095
std_time_shift_dst  = 262.8986519197772